Firstly, let's import every package we will need in this analysis. It is required to install 
```Python
%pip install atlasopenmagic
```
in order to open ATLAS Open Data

In [ ]:
import numpy as np
import pandas as pd                 
import uproot                       # to open .root files
import awkward as ak                # to read data with uproot
#import awkward_pandas
import matplotlib.pyplot as plt     # to plot
import os                           # to manage directories
import random                       # extract random numbers
import requests                     # for HTTP access
import aiohttp                      # HTTP client support
import atlasopenmagic as atom       # to access ATLAS Open Data directly

## Import Data
Now we can download the dataset, selecting the release of interest. It is possible to see every release using
```Python
atom.available_releases()
```
We will use the 2025 release of data taken at $\sqrt{s}=13$ TeV in p-p collisions


In [119]:
atom.set_release("2025e-13tev-beta")    # select release
skim = "GamGam"                         # select skim: events with 2 photons
random.seed(24)                         # set seed for random extractions

# get keys for every dataset: Run2 datas have key="data", MC simulations have key=numbers
all_keys = atom.available_datasets()

data_url_list = atom.get_urls("data", skim, protocol="root", cache=True)      # get list of urls for Run2 data

# get list of urls for MC simulations
mc_url_list = []
for key in all_keys:
    if key != "data":
        mc_url_list += atom.get_urls(key, skim, protocol="https", cache=True)


print(f"Number of MonteCarlo simulated datasets: {len(mc_url_list)}")
print(f"Number of Run2 datasets: {len(data_url_list)}")

Release '2025e-13tev-beta' already active with cached metadata.
Active release: 2025e-13tev-beta. (Datasets path: REMOTE)


Number of MonteCarlo simulated datasets: 373
Number of Run2 datasets: 16


MC simulated datasets are produced considering only one process in each dataset. To separate signal datasets from background datasets we have to select urls that contain the Higgs boson decay channel of interest: into two photons.
Its notation in *gamgam* or *yy*.

In [120]:
# select signal datasets: may contain "gamgam" or "yy"
signal_url_list = [url for url in mc_url_list if ("gamgam" in url) or ("Hyy" in url) or (("_yy" in url) and ("yyy" not in url))]

# select background datasets: all dataset not taken in signal
bkg_url_list = [url for url in mc_url_list if url not in signal_url_list]

print("Number of Signal datasets: ", len(signal_url_list))
print("Number of Background datasets: ", len(bkg_url_list))

Number of Signal datasets:  20
Number of Background datasets:  353


Now we open data in TTrees. In order to have a way to train rapidly the model, it is possible to switch between a subset of data and the entire dataset with a boolean `bool useall`

In [121]:
useall = False      # True: use all dataset
                    # False: use a subset

Explore content of datasets: TTree and TBranches

In [122]:
# See names of trees and branches
print("Tree name: ", uproot.open(mc_url_list[0]).keys())
print("Branches: ", uproot.open(f"{mc_url_list[0]}:analysis").keys())

Tree name:  ['analysis;1']
Branches:  ['sig_ph', 'n_sig_ph', 'num_events', 'sum_of_weights', 'sum_of_weights_squared', 'xsec', 'kfac', 'filteff', 'TriggerMatch_DILEPTON', 'ScaleFactor_MLTRIGGER', 'ScaleFactor_PILEUP', 'ScaleFactor_FTAG', 'mcWeight', 'channelNumber', 'eventNumber', 'runNumber', 'trigML', 'trigP', 'trigDT', 'trigT', 'trigE', 'trigDM', 'trigDE', 'trigM', 'trigMET', 'ScaleFactor_BTAG', 'ScaleFactor_JVT', 'jet_n', 'jet_pt', 'jet_eta', 'jet_phi', 'jet_e', 'jet_btag_quantile', 'jet_jvt', 'largeRJet_n', 'largeRJet_pt', 'largeRJet_eta', 'largeRJet_phi', 'largeRJet_e', 'largeRJet_m', 'largeRJet_D2', 'jet_pt_jer1', 'jet_pt_jer2', 'ScaleFactor_ELE', 'ScaleFactor_MUON', 'ScaleFactor_LepTRIGGER', 'ScaleFactor_MuTRIGGER', 'ScaleFactor_ElTRIGGER', 'lep_n', 'lep_type', 'lep_pt', 'lep_eta', 'lep_phi', 'lep_e', 'lep_charge', 'lep_ptvarcone30', 'lep_topoetcone20', 'lep_z0', 'lep_d0', 'lep_d0sig', 'lep_isTightID', 'lep_isMediumID', 'lep_isLooseID', 'lep_isTightIso', 'lep_isLooseIso', 'lep_

Define TTree name and TBranches that we want to extract for this analysis

In [134]:
tree = "analysis"           # name of TTree in ATLAS OD

features = [branch for branch in (uproot.open(f"{mc_url_list[0]}:{tree}").keys()) 
            if ("photon_" in branch) and ("truth" not in branch)]

#if not useall:
#        mc_ind = set(random.sample(range(len(mc_url_list)), 16))
#        mc_url_list = [url for i, url in enumerate(mc_url_list) if i in mc_ind]
        


Read datasets into awkward arrays. This procedure avoids errors in calling CERN server and it is the faster solution.
Then, label=1 is assigned to signal events, label=0 is assigned to background events.

**This download may last up to 15-20 minutes!**
Then, it is possible to save locally the awkward arrays and import them rapidly, without download

In [ ]:
# read signal datasets into awkward array
signal_array = []
for url in signal_url_list:    
    with uproot.open(f"{url}:{tree}") as tree_opened:
        arr = tree_opened.arrays(filter_name=features, library="ak")
        signal_array.append(arr)
signal_awk = ak.concatenate(signal_array)

# assign label 1 to signal
signal_awk["label"] = 1


# read background datasets into awkward array
bkg_array = []
for url in bkg_url_list:
    with uproot.open(f"{url}:{tree}") as tree_opened:
        arr = tree_opened.arrays(filter_name=features, library="ak")
        bkg_array.append(arr)
bkg_awk = ak.concatenate(bkg_array)

# assign label 0 to background
bkg_awk["label"] = 0            


# merge signal and background arrays into one
mc_awk = ak.concatenate([signal_awk, bkg_awk])

# create directory if needed and save array locally
os.makedirs("data", exist_ok=True)
ak.to_parquet(mc_awk, "data/mc_awkward_complete.parquet") 
print("Awkward array correctly saved!")

In [ ]:
# recover array from local memory
mc_awk = ak.from_parquet("data/mc_awkward_complete.parquet")

print("Awkward array correctly recovered!")

In [ ]:
# preview
ak.to_dataframe(signal_awk[:10])

photon_n  photon_pt  photon_eta  photon_phi    photon_e  \
entry subentry                                                            
0     0                2  55.753937    2.183393   -1.649362  250.586884   
      1                2  44.155506    0.532499    0.756613   50.565098   
1     0                2  87.744835    0.929985   -0.068314  128.503632   
      1                2  44.386959    1.309105   -2.791158   88.172897   
2     0                2  72.883636   -1.694380   -1.193044  205.057480   
      1                2  42.663708   -0.634014    1.493021   51.529682   
3     0                2  63.968410   -1.117329   -2.653243  108.229164   
      1                2  60.006382   -1.290419    0.489502  117.296410   
4     0                2  71.630692   -1.959487   -1.995904  259.181885   
      1                2  53.195244   -1.906264    1.277885  182.899567   
5     0                2  84.709709   -0.007789   -1.595317   84.712280   
      1                2  28.937256   -1.340593    2.093522   59.075417   
6     0                2  65.865799    0.050281    0.479313   65.949074   
      1                2  56.499058    0.414607   -2.647204   61.425095   
7     0                2  62.731876    0.227773   -1.582608   64.366203   
      1                2  53.909336   -0.854986    2.263641   74.843178   
8     0                2  60.498058   -0.755705   -1.282761   78.610893   
      1                2  48.412071   -1.883282    2.425446  162.837341   
9     0                2  67.375938    1.840729   -2.535791  217.618515   
      1                2  34.925453    0.353146    1.180826   37.125992   

                photon_ptcone20  photon_topoetcone40  photon_isLooseID  \
entry subentry                                                           
0     0                     0.0             3.950014              True   
      1                     0.0             0.762171              True   
1     0                     0.0            -0.756278              True   
      1                     0.0            -1.453551              True   
2     0                     0.0            -0.734271              True   
      1                     0.0            -1.213254              True   
3     0                     0.0             0.505048              True   
      1                     0.0            -1.683037              True   
4     0                     0.0            -0.460714              True   
      1                     0.0            -1.059433              True   
5     0                     0.0             0.416635              True   
      1                     0.0             0.642250              True   
6     0                     0.0             5.444521              True   
      1                     0.0            -1.637449              True   
7     0                     0.0             0.526252              True   
      1                     0.0             3.779907              True   
8     0                     0.0            -0.192389              True   
      1                     0.0            -1.682197              True   
9     0                     0.0             0.551040              True   
      1                     0.0             3.953416              True   

                photon_isTightID  photon_isLooseIso  photon_isTightIso  label  
entry subentry                                                                 
0     0                     True               True              False      1  
      1                     True              False               True      1  
1     0                     True               True               True      1  
      1                     True               True               True      1  
2     0                     True               True               True      1  
      1                     True               True               True      1  
3     0                     True               True               True      1  
      1         